# 10장. 비전 트랜스포머 (Vision Transformer)
> **실습 데이터셋**: FashionMNIST (10클래스)  
> **모델**: ViT → Swin Transformer → CvT


## 1. ViT (Vision Transformer)
### 예제 10.1 — FashionMNIST 다운로드 및 서브샘플링


In [1]:
from itertools import chain
from collections import defaultdict
from torch.utils.data import Subset
from torchvision import datasets

def subset_sampler(dataset, classes, max_len):
    target_idx = defaultdict(list)
    for idx, label in enumerate(dataset.targets):
        target_idx[int(label)].append(idx)
    indices = list(
        chain.from_iterable(
            [target_idx[idx][:max_len] for idx in range(len(classes))]
        )
    )
    return Subset(dataset, indices)

train_dataset = datasets.FashionMNIST(root="./datasets", download=True, train=True)
test_dataset  = datasets.FashionMNIST(root="./datasets", download=True, train=False)
classes       = train_dataset.classes
class_to_idx  = train_dataset.class_to_idx

print(classes)
print(class_to_idx)

subset_train_dataset = subset_sampler(train_dataset, classes, max_len=1000)
subset_test_dataset  = subset_sampler(test_dataset,  classes, max_len=100)

print(f"Training Data Size : {len(subset_train_dataset)}")
print(f"Testing Data Size  : {len(subset_test_dataset)}")
print(train_dataset[0])


['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
{'T-shirt/top': 0, 'Trouser': 1, 'Pullover': 2, 'Dress': 3, 'Coat': 4, 'Sandal': 5, 'Shirt': 6, 'Sneaker': 7, 'Bag': 8, 'Ankle boot': 9}
Training Data Size : 10000
Testing Data Size  : 1000
(<PIL.Image.Image image mode=L size=28x28 at 0x...>, 9)


### 예제 10.2 — 이미지 전처리
28×28 흑백(1ch) → 224×224 RGB(3ch) 변환 + 정규화


In [2]:
import torch
from torchvision import transforms
from transformers import AutoImageProcessor

MODEL_NAME_VIT = "google/vit-base-patch16-224-in21k"

image_processor = AutoImageProcessor.from_pretrained(MODEL_NAME_VIT)

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((image_processor.size["height"],
                       image_processor.size["width"])),
    transforms.Lambda(lambda x: torch.cat([x, x, x], dim=0)),  # 1ch → 3ch
    transforms.Normalize(mean=image_processor.image_mean,
                         std=image_processor.image_std),
])

print(f"size : {image_processor.size}")
print(f"mean : {image_processor.image_mean}")
print(f"std  : {image_processor.image_std}")


size : {'height': 224, 'width': 224}
mean : (0.5, 0.5, 0.5)
std  : (0.5, 0.5, 0.5)


### 예제 10.3 — ViT DataLoader
ViT 입력 형식: `{"pixel_values": Tensor, "labels": Tensor}`


In [3]:
from torch.utils.data import DataLoader

def collator(batch, transform):
    images, labels = zip(*batch)
    pixel_values = torch.stack([transform(img) for img in images])
    labels = torch.tensor(labels)
    return {"pixel_values": pixel_values, "labels": labels}

train_dataloader = DataLoader(
    subset_train_dataset, batch_size=32, shuffle=True,
    collate_fn=lambda b: collator(b, transform), drop_last=True,
)
valid_dataloader = DataLoader(
    subset_test_dataset, batch_size=4, shuffle=False,
    collate_fn=lambda b: collator(b, transform), drop_last=True,
)

batch = next(iter(train_dataloader))
for k, v in batch.items():
    print(f"{k} : {v.shape}")


pixel_values : torch.Size([32, 3, 224, 224])
labels : torch.Size([32])


### 예제 10.4 — 사전 학습된 ViT 모델
`ignore_mismatched_sizes=True` 로 분류기를 10 클래스로 교체


In [4]:
from transformers import ViTForImageClassification

model = ViTForImageClassification.from_pretrained(
    MODEL_NAME_VIT,
    num_labels=len(classes),
    id2label={idx: label for label, idx in class_to_idx.items()},
    label2id=class_to_idx,
    ignore_mismatched_sizes=True,
)
print(model.classifier)


Linear(in_features=768, out_features=10, bias=True)


### 예제 10.5 — 패치 임베딩 구조 확인
224×224, 16×16 패치 → 196개 + [CLS] = 197


In [5]:
print(model.vit.embeddings)

batch = next(iter(train_dataloader))
print("image shape            :", batch["pixel_values"].shape)
print("patch embeddings shape :",
      model.vit.embeddings.patch_embeddings(batch["pixel_values"]).shape)
print("[CLS]+patch emb shape  :",
      model.vit.embeddings(batch["pixel_values"]).shape)


ViTEmbeddings(
  (patch_embeddings): ViTPatchEmbeddings(
    (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  )
  (dropout): Dropout(p=0.0, inplace=False)
)
image shape            : torch.Size([32, 3, 224, 224])
patch embeddings shape : torch.Size([32, 196, 768])
[CLS]+patch emb shape  : torch.Size([32, 197, 768])


### 예제 10.6 — 하이퍼파라미터 설정


In [6]:
from transformers import TrainingArguments

args = TrainingArguments(
    output_dir="./models/ViT-FashionMNIST",
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.001,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=125,
    remove_unused_columns=False,
    seed=7,
)


### 예제 10.7 — 매크로 평균 F1 점수


In [7]:
import evaluate
import numpy as np

def compute_metrics(eval_pred):
    metric = evaluate.load("f1")
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions,
                          references=labels, average="macro")


### 예제 10.8 — ViT 전체 학습 코드
> 책 기준 3 epoch (클래스당 1,000장) 기준 최종 F1: **0.9231**


In [8]:
from transformers import Trainer

def model_init(classes, class_to_idx):
    return ViTForImageClassification.from_pretrained(
        MODEL_NAME_VIT,
        num_labels=len(classes),
        id2label={idx: label for label, idx in class_to_idx.items()},
        label2id=class_to_idx,
        ignore_mismatched_sizes=True,
    )

trainer = Trainer(
    model_init=lambda _: model_init(classes, class_to_idx),
    args=args,
    train_dataset=subset_train_dataset,
    eval_dataset=subset_test_dataset,
    data_collator=lambda b: collator(b, transform),
    compute_metrics=compute_metrics,
    processing_class=image_processor,
)
trainer.train()


{'loss': 0.7062, 'epoch': 1.0}  {'eval_loss': 0.6377, 'eval_f1': 0.8905, 'epoch': 1.0}
{'loss': 0.4954, 'epoch': 2.0}  {'eval_loss': 0.4781, 'eval_f1': 0.9166, 'epoch': 2.0}
{'loss': 0.4078, 'epoch': 3.0}  {'eval_loss': 0.4357, 'eval_f1': 0.9231, 'epoch': 3.0}


### 예제 10.9 — ViT 성능 평가 (혼동 행렬)


In [9]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

outputs = trainer.predict(subset_test_dataset)
print(outputs.metrics)

y_true = outputs.label_ids
y_pred = outputs.predictions.argmax(1)
matrix = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=list(classes))
_, ax = plt.subplots(figsize=(10, 10))
disp.plot(xticks_rotation=45, ax=ax)
plt.title("ViT — FashionMNIST 혼동 행렬")
plt.tight_layout()
plt.show()


{'test_loss': 0.43577, 'test_f1': 0.9232, 'test_runtime': 8.4, 'test_samples_per_second': 29.6}


---
## 2. Swin Transformer
### 예제 10.10 — 상대적 위치 편향: 좌표 계산


In [10]:
import torch

window_size = 2
coords_h = torch.arange(window_size)
coords_w = torch.arange(window_size)
coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing="ij"))
coords_flatten = torch.flatten(coords, 1)

relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
print(relative_coords)
print("shape:", relative_coords.shape)


tensor([[[ 0,  0, -1, -1],
         [ 0,  0, -1, -1],
         [ 1,  1,  0,  0],
         [ 1,  1,  0,  0]],

        [[ 0, -1,  0, -1],
         [ 1,  0,  1,  0],
         [ 0, -1,  0, -1],
         [ 1,  0,  1,  0]]])
shape: torch.Size([2, 4, 4])


### 예제 10.11 — X·Y축 위치 행렬


In [11]:
x_coords = relative_coords[0].clone()
y_coords = relative_coords[1].clone()

x_coords += window_size - 1          # 음수 제거 (오프셋)
y_coords += window_size - 1
x_coords *= 2 * window_size - 1      # X에 Y 범위 곱해 고유 인덱스 생성

print(f"X에 대한 행렬:\n{x_coords}\n")
print(f"Y에 대한 행렬:\n{y_coords}\n")


X에 대한 행렬:
tensor([[3, 3, 0, 0],
        [3, 3, 0, 0],
        [6, 6, 3, 3],
        [6, 6, 3, 3]])

Y에 대한 행렬:
tensor([[1, 0, 1, 0],
        [2, 1, 2, 1],
        [1, 0, 1, 0],
        [2, 1, 2, 1]])



### 예제 10.12 — 상대적 위치 편향 테이블
`Attention(Q,K,V) = Softmax(QKᵀ/√d + B)V`


In [12]:
relative_position_index = x_coords + y_coords
print(f"X·Y축 위치 인덱스:\n{relative_position_index}")

num_heads = 1
table_size = (2 * window_size - 1) * (2 * window_size - 1)
relative_position_bias_table = torch.zeros(table_size, num_heads)

relative_position_bias = relative_position_bias_table[
    relative_position_index.view(-1)
].view(window_size * window_size, window_size * window_size, -1)

print("상대적 위치 편향 shape:", relative_position_bias.shape)


X·Y축 위치 인덱스:
tensor([[4, 3, 1, 0],
        [5, 4, 2, 1],
        [7, 6, 4, 3],
        [8, 7, 5, 4]])
상대적 위치 편향 shape: torch.Size([4, 4, 1])


### 예제 10.13 — 사전 학습된 Swin Transformer 모델


In [13]:
from transformers import SwinForImageClassification, AutoImageProcessor

MODEL_NAME_SWIN = "microsoft/swin-tiny-patch4-window7-224"
image_processor_swin = AutoImageProcessor.from_pretrained(MODEL_NAME_SWIN)

model_swin = SwinForImageClassification.from_pretrained(
    MODEL_NAME_SWIN,
    num_labels=len(classes),
    id2label={idx: label for label, idx in class_to_idx.items()},
    label2id=class_to_idx,
    ignore_mismatched_sizes=True,
)

for main_name, main_module in model_swin.named_children():
    print(main_name)
    for sub_name, _ in main_module.named_children():
        print(" ", sub_name)


swin
  embeddings
  encoder
  layernorm
  pooler
classifier


### 예제 10.14 — Swin 패치 임베딩
224×224, 4×4 커널·stride 4 → 56×56 = 3,136 패치


In [14]:
transform_swin = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((image_processor_swin.size["height"],
                       image_processor_swin.size["width"])),
    transforms.Lambda(lambda x: torch.cat([x, x, x], dim=0)),
    transforms.Normalize(mean=image_processor_swin.image_mean,
                         std=image_processor_swin.image_std),
])

train_dataloader_swin = DataLoader(
    subset_train_dataset, batch_size=32, shuffle=True,
    collate_fn=lambda b: collator(b, transform_swin), drop_last=True,
)

batch = next(iter(train_dataloader_swin))
print("이미지 차원:", batch["pixel_values"].shape)

patch_emb_output, _ = model_swin.swin.embeddings.patch_embeddings(batch["pixel_values"])
print("모듈:", model_swin.swin.embeddings.patch_embeddings)
print("패치 임베딩 차원:", patch_emb_output.shape)


이미지 차원: torch.Size([32, 3, 224, 224])
모듈: SwinPatchEmbeddings(
  (projection): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
)
패치 임베딩 차원: torch.Size([32, 3136, 96])


### 예제 10.15 — Swin 블록 구조


In [15]:
for name, module in model_swin.swin.encoder.layers[0].named_children():
    print(name)
    for sub_name, _ in module.named_children():
        print(" ", sub_name)


blocks
  0
  1
downsample
  reduction
  norm


### 예제 10.16 — SwinLayer 상세 구조
계층 정규화 → W-MSA(또는 SW-MSA) → 계층 정규화 → MLP(GELU)


In [16]:
print(model_swin.swin.encoder.layers[0].blocks[0])


SwinLayer(
  (layernorm_before): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  (attention): SwinAttention(
    (self): SwinSelfAttention(
      (query): Linear(in_features=96, out_features=96, bias=True)
      (key): Linear(in_features=96, out_features=96, bias=True)
      (value): Linear(in_features=96, out_features=96, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (output): SwinSelfOutput(
      (dense): Linear(in_features=96, out_features=96, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (drop_path): SwinDropPath(p=0.1)
  (layernorm_after): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
  (intermediate): SwinIntermediate(
    (dense): Linear(in_features=96, out_features=384, bias=True)
    (intermediate_act_fn): GELUActivation()
  )
  (output): SwinOutput(
    (dense): Linear(in_features=384, out_features=96, bias=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
)


### 예제 10.17 — W-MSA · SW-MSA 실행


In [17]:
W_MSA  = model_swin.swin.encoder.layers[0].blocks[0]   # Window MSA
SW_MSA = model_swin.swin.encoder.layers[0].blocks[1]   # Shifted-Window MSA

W_MSA_output  = W_MSA(patch_emb_output,  W_MSA.input_resolution)[0]
SW_MSA_output = SW_MSA(W_MSA_output, SW_MSA.input_resolution)[0]

print("패치 임베딩 차원:", patch_emb_output.shape)
print("W-MSA  결과 차원:", W_MSA_output.shape)
print("SW-MSA 결과 차원:", SW_MSA_output.shape)


패치 임베딩 차원: torch.Size([32, 3136, 96])
W-MSA  결과 차원: torch.Size([32, 3136, 96])
SW-MSA 결과 차원: torch.Size([32, 3136, 96])


### 예제 10.18 — 패치 병합
3,136(56×56)패치 → 784(28×28)패치, 채널 96→192


In [18]:
patch_merge = model_swin.swin.encoder.layers[0].downsample
print("patch_merge 모듈:", patch_merge)

output = patch_merge(SW_MSA_output, patch_merge.input_resolution)
print("patch_merge 결과 차원:", output.shape)


patch_merge 모듈: SwinPatchMerging(
  (reduction): Linear(in_features=384, out_features=192, bias=False)
  (norm): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
)
patch_merge 결과 차원: torch.Size([32, 784, 192])


### 예제 10.19 — Swin Transformer 학습
> 책 기준 3 epoch 최종 F1: **0.9159**, 검증 손실 0.2416


In [19]:
args_swin = TrainingArguments(
    output_dir="./models/Swin-FashionMNIST",
    save_strategy="epoch", eval_strategy="epoch",
    learning_rate=1e-5, per_device_train_batch_size=16,
    per_device_eval_batch_size=16, num_train_epochs=3,
    weight_decay=0.001, load_best_model_at_end=True,
    metric_for_best_model="f1", logging_steps=125,
    remove_unused_columns=False, seed=7,
)

def model_init_swin(classes, class_to_idx):
    return SwinForImageClassification.from_pretrained(
        MODEL_NAME_SWIN,
        num_labels=len(classes),
        id2label={idx: label for label, idx in class_to_idx.items()},
        label2id=class_to_idx,
        ignore_mismatched_sizes=True,
    )

trainer_swin = Trainer(
    model_init=lambda _: model_init_swin(classes, class_to_idx),
    args=args_swin,
    train_dataset=subset_train_dataset,
    eval_dataset=subset_test_dataset,
    data_collator=lambda b: collator(b, transform_swin),
    compute_metrics=compute_metrics,
    processing_class=image_processor_swin,
)
trainer_swin.train()


{'loss': 0.3720, 'epoch': 1.0}  {'eval_loss': 0.3046, 'eval_f1': 0.8985, 'epoch': 1.0}
{'loss': 0.2579, 'epoch': 2.0}  {'eval_loss': 0.2688, 'eval_f1': 0.9138, 'epoch': 2.0}
{'loss': 0.2607, 'epoch': 3.0}  {'eval_loss': 0.2416, 'eval_f1': 0.9159, 'epoch': 3.0}


---
## 3. CvT (Convolutional Vision Transformer)
### 예제 10.20 — CvT 이미지 전처리
ViT·Swin은 `height/width`, CvT는 `shortest_edge` 사용


In [20]:
MODEL_NAME_CVT = "microsoft/cvt-21"
image_processor_cvt = AutoImageProcessor.from_pretrained(MODEL_NAME_CVT)

transform_cvt = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((image_processor_cvt.size["shortest_edge"],
                       image_processor_cvt.size["shortest_edge"])),
    transforms.Lambda(lambda x: torch.cat([x, x, x], dim=0)),
    transforms.Normalize(mean=image_processor_cvt.image_mean,
                         std=image_processor_cvt.image_std),
])
print("size:", image_processor_cvt.size)


size: {'shortest_edge': 224}


### 예제 10.21 — 사전 학습된 CvT 모델
3개 스테이지 구조, 풀링 없음


In [21]:
from transformers import CvtForImageClassification

model_cvt = CvtForImageClassification.from_pretrained(
    MODEL_NAME_CVT,
    num_labels=len(classes),
    id2label={idx: label for label, idx in class_to_idx.items()},
    label2id=class_to_idx,
    ignore_mismatched_sizes=True,
)

for main_name, main_module in model_cvt.named_children():
    print(main_name)
    for sub_name, _ in main_module.named_children():
        print(" ", sub_name)


cvt
  encoder
layernorm
classifier


### 예제 10.22 — CvT 스테이지 구조
스테이지 0: Conv 7×7 stride 4 (224→56)


In [22]:
stages = model_cvt.cvt.encoder.stages
print(stages[0])


CvtStage(
  (embedding): CvtEmbeddings(
    (convolution_embeddings): CvtConvEmbeddings(
      (projection): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(2, 2))
      (normalization): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (layers): Sequential(...)
)


### 예제 10.23 — CvT 셀프 어텐션 적용
패치 임베딩 [B,64,56,56] → [B,3136,64] → 셀프 어텐션


In [23]:
train_dataloader_cvt = DataLoader(
    subset_train_dataset, batch_size=32, shuffle=True,
    collate_fn=lambda b: collator(b, transform_cvt), drop_last=True,
)

batch = next(iter(train_dataloader_cvt))
print("이미지 차원:", batch["pixel_values"].shape)

patch_emb_cvt = stages[0].embedding(batch["pixel_values"])
print("패치 임베딩 차원:", patch_emb_cvt.shape)

B, C, H, W = patch_emb_cvt.shape
hidden = patch_emb_cvt.view(B, C, H * W).permute(0, 2, 1)
print("셀프 어텐션 입력 차원:", hidden.shape)

attn_output = stages[0].layers[0].attention.attention(hidden, H, W)
print("셀프 어텐션 출력 차원:", attn_output.shape)


이미지 차원: torch.Size([32, 3, 224, 224])
패치 임베딩 차원: torch.Size([32, 64, 56, 56])
셀프 어텐션 입력 차원: torch.Size([32, 3136, 64])
셀프 어텐션 출력 차원: torch.Size([32, 3136, 64])


### 예제 10.24 — CvT 학습
> Apple Silicon MPS 환경에서는 첫 줄 `torch.backends.mps.is_available = lambda: False` 필수
> 책 기준 3 epoch 최종 F1: **0.9173**


In [24]:
import torch
# Apple Silicon MPS에서 CvT backward 버그 우회 (CPU로 강제)
torch.backends.mps.is_available = lambda: False

args_cvt = TrainingArguments(
    output_dir="./models/CvT-FashionMNIST",
    save_strategy="epoch", eval_strategy="epoch",
    learning_rate=1e-5, per_device_train_batch_size=16,
    per_device_eval_batch_size=16, num_train_epochs=3,
    weight_decay=0.001, load_best_model_at_end=True,
    metric_for_best_model="f1", logging_steps=125,
    remove_unused_columns=False, seed=7,
)

def model_init_cvt(classes, class_to_idx):
    return CvtForImageClassification.from_pretrained(
        MODEL_NAME_CVT,
        num_labels=len(classes),
        id2label={idx: label for label, idx in class_to_idx.items()},
        label2id=class_to_idx,
        ignore_mismatched_sizes=True,
    )

trainer_cvt = Trainer(
    model_init=lambda _: model_init_cvt(classes, class_to_idx),
    args=args_cvt,
    train_dataset=subset_train_dataset,
    eval_dataset=subset_test_dataset,
    data_collator=lambda b: collator(b, transform_cvt),
    compute_metrics=compute_metrics,
    processing_class=image_processor_cvt,
)
trainer_cvt.train()


{'loss': 0.7737, 'epoch': 1.0}  {'eval_loss': 0.3269, 'eval_f1': 0.8914, 'epoch': 1.0}
{'loss': 0.7008, 'epoch': 2.0}  {'eval_loss': 0.2596, 'eval_f1': 0.9172, 'epoch': 2.0}
{'loss': 0.6438, 'epoch': 3.0}  {'eval_loss': 0.2460, 'eval_f1': 0.9173, 'epoch': 3.0}
